# Stock Market AI Bot - Colab Walk-Forward

This notebook runs the next **lower-turnover nested walk-forward** on Google Colab. It is research-only: it does not publish a live configuration and it cannot place trades. Checkpoints and final results stay in Google Drive.

## 1. Connect Google Drive
Google will ask you to approve access. The notebook needs Drive only to read the project bundle and preserve progress.

In [ ]:
# Google Colab provides this helper for connecting your own Drive.
from google.colab import drive
drive.mount('/content/drive')

## 2. Choose the run settings
The defaults are deliberately cautious for a normal Colab CPU runtime. Use a High-RAM runtime before trying 4 workers. Keep `RESET_RUN` false when resuming after a disconnect.

In [ ]:
from pathlib import Path

# The zip file created by prepare_colab_walkforward.py goes here.
DRIVE_HOME = Path('/content/drive/MyDrive/StockBotColab')
BUNDLE_PATH = DRIVE_HOME / 'Stock_Market_AI_Bot_Colab.zip'

# A new RUN_NAME creates an independent result and checkpoint folder.
RUN_NAME = 'low_turnover_next'
GRID_MODE = 'low-turnover'
WORKERS = 2
LOW_MEMORY = True
RESET_RUN = False

assert GRID_MODE in {'low-turnover', 'stable', 'recent-alpha', 'adaptive-sizing', 'balanced', 'full'}
assert WORKERS >= 1
print('Bundle:', BUNDLE_PATH)
print('Drive run folder:', DRIVE_HOME / 'runs' / RUN_NAME)

## 3. Prepare fast local files and persistent progress
The project runs from Colab's local disk for speed. Its small `signals` folder is linked to Drive so each finished yearly fold is preserved.

In [ ]:
import os
import shutil
import zipfile

if not BUNDLE_PATH.exists():
    raise FileNotFoundError(f'Upload the project bundle here first: {BUNDLE_PATH}')

extract_root = Path('/content/stockbot_bundle')
project_dir = Path('/content/stockbot')
drive_run_dir = DRIVE_HOME / 'runs' / RUN_NAME
drive_signals = drive_run_dir / 'signals'

# RESET_RUN deletes only this named Colab run, never the uploaded bundle.
if RESET_RUN and drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)

shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(project_dir, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as bundle:
    bundle.extractall(extract_root)

# Find the extracted project by its walk-forward entry point.
entry_points = list(extract_root.rglob('core_satellite_nested_walkforward.py'))
if len(entry_points) != 1:
    raise RuntimeError(f'Expected one project in the bundle, found {len(entry_points)}')
source_project = entry_points[0].parent
shutil.copytree(source_project, project_dir)

# On the first run, seed Drive with the signal inputs from the bundle.
drive_signals.mkdir(parents=True, exist_ok=True)
local_signals = project_dir / 'signals'
for source_file in local_signals.glob('*'):
    destination = drive_signals / source_file.name
    if source_file.is_file() and not destination.exists():
        shutil.copy2(source_file, destination)

# Replace the temporary signals folder with a link to persistent Drive.
shutil.rmtree(local_signals)
os.symlink(drive_signals, local_signals, target_is_directory=True)
print('Project ready:', project_dir)
print('Persistent signals:', drive_signals)

## 4. Install the project's Python packages
This can take several minutes on a fresh runtime. Colab may show package warnings while it replaces its preinstalled versions.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(project_dir / 'requirements.txt')],
    check=True,
)
print('Packages installed')

## 5. Verify the uploaded research data
This catches a missing or incomplete bundle before the long calculation starts.

In [ ]:
import pandas as pd

data_files = sorted((project_dir / 'data').glob('*.parquet'))
required_benchmarks = [project_dir / 'data' / 'SPY.parquet', project_dir / 'data' / 'QQQ.parquet']
missing = [str(path) for path in required_benchmarks if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing benchmark data: {missing}')
if len(data_files) < 10:
    raise RuntimeError(f'Only {len(data_files)} Parquet files found; the bundle looks incomplete')

for benchmark in required_benchmarks:
    frame = pd.read_parquet(benchmark)
    date_values = pd.to_datetime(frame.index if not isinstance(frame.index, pd.RangeIndex) else frame.get('date'), errors='coerce')
    print(benchmark.name, 'latest date:', date_values.max())
print('Parquet files found:', len(data_files))

## 6. Run the nested walk-forward
Leave this cell running. The batch wrapper starts a fresh process for each yearly fold, which limits memory growth. Results stay research-only because `--no-publish-live-config` is always used.

In [ ]:
grid_flags = {
    'low-turnover': ['--low-turnover-grid'],
    'stable': ['--stable-grid'],
    'recent-alpha': ['--recent-alpha-grid'],
    'adaptive-sizing': ['--adaptive-sizing-grid'],
    'balanced': [],
    'full': ['--full'],
}
output_prefix = f'colab_{RUN_NAME}'
command = [
    sys.executable,
    'run_walkforward_batched.py',
    *grid_flags[GRID_MODE],
    '--strategy', 'core-alpha',
    '--workers', str(WORKERS),
    '--max-batches', '30',
    '--output-prefix', output_prefix,
    '--no-publish-live-config',
    '--low-memory' if LOW_MEMORY else '--no-low-memory',
]

# Limit hidden math-library threads so worker processes do not fight each other.
run_environment = os.environ.copy()
run_environment.update({
    'OMP_NUM_THREADS': '1',
    'OPENBLAS_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'NUMEXPR_NUM_THREADS': '1',
})
print('Starting research-only run:', ' '.join(command))
subprocess.run(command, cwd=project_dir, env=run_environment, check=True)
print('Walk-forward finished. Results are in:', drive_signals)

## 7. Show the saved result files
Download the newest JSON and CSV files for review in the local project. Do not treat a completed calculation as trading approval.

In [ ]:
result_files = sorted(
    list(drive_signals.glob(f'{output_prefix}*.json')) +
    list(drive_signals.glob(f'{output_prefix}*.csv')),
    key=lambda path: path.stat().st_mtime,
)
if not result_files:
    raise FileNotFoundError('No final JSON/CSV found. Check the run cell output and checkpoint.')
for result_file in result_files:
    print(result_file)